<a href="https://colab.research.google.com/github/mediolanum1/dla/blob/2024/CTC_WER_CER_CTC_with_beam_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install pickle
!pip install torch
!pip install editdistance

ERROR: Could not find a version that satisfies the requirement pickle (from versions: none)
ERROR: No matching distribution found for pickle


In [6]:
import torch
import pickle

# CTC decoding

In [7]:
with open("mystery_records.pickle", "rb") as f: # loading pre-computed CTC output
    batch = pickle.load(f)

log_probs = batch["log_probs"] # log probs of softmax layer [batch_size, T, vocab_size]

ind2char = batch["ind2char"] # dict with index to char mapping

EMPTY_TOK = "^"
EMPTY_IND = 0

In [9]:
def ctc_decode(inds, ind2char):
  decoded = []
  last_char_ind = EMPTY_IND
  for ind in inds:
    if last_char_ind == ind:
      continue
    if ind != EMPTY_IND:
      decoded.append(ind2char[ind])
    last_char_ind = ind

  return "".join(decoded)

for i, rec in enumerate(log_probs):
  text = ctc_decode(rec.argmax(-1).numpy(), ind2char) # here argmax takes from log-probs highest value
  print(f"{i}) {text}")

0) we nostrngesto love you know therols and so do i a foll commitment what i thinking of you wolden get this from any ather guy
1)  never gona give you up never donelet you down never go arun around and deset you never gon a make you cri never gonna say good by


# CER, WER metrics

Reminder:

WER - word error rate - number of insertions, deletions and substitutions to get target from prediction. Considered to be main metric

CER - character error rate - same as WER but at char level

Could be the case when WER is high and CER is low and vice versa

In [15]:
import editdistance # library for fast calc of edit distance

def calc_wer(target_text: str, pred_text: str):
  return editdistance.eval(target_text.split(), pred_text.split()) / len(target_text.split())

def calc_cer(target_text: str, pred_text: str):
  return editdistance.eval(target_text, pred_text) / len(target_text)

In [26]:
import numpy as np

for target, pred, expected_wer, expected_cer in [
  ("if you can not measure it you can not improve it",
     "if you can nt measure t yo can not i",
     0.454, 0.25),
    ("if you cant describe what you are doing as a process you dont know what youre doing",
     "if you cant describe what you are doing as a process you dont know what youre doing",
     0.0, 0.0),
    ("one measurement is worth a thousand expert opinions",
     "one  is worth thousand opinions",
     0.375, 0.392),
    ("Well, everybody's talkin' 'bout the stormy weather And what's a man to do",
    "We will, every bundle stalking boat short and winter andy watt amen so we aredo ", 0,0)
]:
    wer = calc_wer(target, pred)
    cer = calc_cer(target, pred)
    print(cer, " ", expected_cer, " ------ ", wer , " ", expected_wer)
    assert np.isclose(wer, expected_wer, atol=1e-3), f"true: {target}, pred: {pred}, expected wer {expected_wer} != your wer {wer}"
    assert np.isclose(cer, expected_cer, atol=1e-3), f"true: {target}, pred: {pred}, expected cer {expected_cer} != your cer {cer}"


0.25   0.25  ------  0.45454545454545453   0.454
0.0   0.0  ------  0.0   0.0
0.39215686274509803   0.392  ------  0.375   0.375
0.5342465753424658   0  ------  1.1538461538461537   0


AssertionError: true: Well, everybody's talkin' 'bout the stormy weather And what's a man to do, pred: We will, every bundle stalking boat short and winter andy watt amen so we aredo , expected wer 0 != your wer 1.1538461538461537